In [2]:
# Week 3 Day 6 — U-Net Architecture
# Goal: Build complete U-Net with Dice Loss

import torch
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print("✓ Setup complete")

Device: cuda
✓ Setup complete


In [3]:
# Cell 2 — Encoder and Decoder blocks
# Same classes we already tested in Day 5

import torch
import torch.nn as nn

class EncoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.relu  = nn.ReLU()
        self.pool  = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        skip = x
        x = self.pool(x)
        return x, skip


class DecoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv1 = nn.Conv2d(in_channels * 2, out_channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.relu  = nn.ReLU()

    def forward(self, x, skip):
        x = self.upsample(x)
        x = torch.cat([x, skip], dim=1)
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        return x

print("✓ EncoderBlock and DecoderBlock classes defined")

✓ EncoderBlock and DecoderBlock classes defined


In [5]:
# Cell 3 — Complete U-Net
# Encoder (4 levels) + Bottleneck + Decoder (4 levels)

class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()

        # ── Encoder — walking away from the beach ──
        self.enc1 = EncoderBlock(in_channels, 16)   # 1   → 16
        self.enc2 = EncoderBlock(16, 32)             # 16  → 32
        self.enc3 = EncoderBlock(32, 64)             # 32  → 64
        self.enc4 = EncoderBlock(64, 128)            # 64  → 128

        # ── Bottleneck — deepest, most compressed point ──
        self.bottleneck_conv1 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bottleneck_conv2 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.relu = nn.ReLU()

        # ── Decoder — walking back toward the beach ──
        self.dec4 = DecoderBlock(256, 128)  # matches bottleneck(256) + skip4(128)...
        self.dec3 = DecoderBlock(128, 64)
        self.dec2 = DecoderBlock(64, 32)
        self.dec1 = DecoderBlock(32, 16)

        # ── Final output layer ──
        self.final_conv = nn.Conv2d(16, out_channels, kernel_size=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # Encoder
        x, skip1 = self.enc1(x)
        print(f"After enc1: x={x.shape}, skip1={skip1.shape}")
        x, skip2 = self.enc2(x)
        print(f"After enc2: x={x.shape}, skip2={skip2.shape}")
        x, skip3 = self.enc3(x)
        print(f"After enc3: x={x.shape}, skip3={skip3.shape}")
        x, skip4 = self.enc4(x)
        print(f"After enc4: x={x.shape}, skip4={skip4.shape}")

        # Bottleneck
        x = self.relu(self.bottleneck_conv1(x))
        x = self.relu(self.bottleneck_conv2(x))
        print(f"After bottleneck: x={x.shape}")

        # Decoder
        x = self.dec4(x, skip4)
        print(f"After dec4: {x.shape}")
        x = self.dec3(x, skip3)
        print(f"After dec3: {x.shape}")
        x = self.dec2(x, skip2)
        print(f"After dec2: {x.shape}")
        x = self.dec1(x, skip1)
        print(f"After dec1: {x.shape}")

        # Final output
        x = self.final_conv(x)
        x = self.sigmoid(x)
        print(f"Final output: {x.shape}")

        return x

print("✓ UNet class defined")

✓ UNet class defined


In [6]:
# Cell 4 — Test the U-Net with a fake CT slice

model = UNet(in_channels=1, out_channels=1).to(device)

# Fake CT slice — small size for quick testing
ct_input = torch.zeros(1, 1, 128, 128).to(device)
print(f"Input shape: {ct_input.shape}\n")

with torch.no_grad():
    output = model(ct_input)

print(f"\n✓ Final segmentation output shape: {output.shape}")

Input shape: torch.Size([1, 1, 128, 128])

After enc1: x=torch.Size([1, 16, 64, 64]), skip1=torch.Size([1, 16, 128, 128])
After enc2: x=torch.Size([1, 32, 32, 32]), skip2=torch.Size([1, 32, 64, 64])
After enc3: x=torch.Size([1, 64, 16, 16]), skip3=torch.Size([1, 64, 32, 32])
After enc4: x=torch.Size([1, 128, 8, 8]), skip4=torch.Size([1, 128, 16, 16])
After bottleneck: x=torch.Size([1, 256, 8, 8])


RuntimeError: Given groups=1, weight of size [128, 512, 3, 3], expected input[1, 384, 16, 16] to have 512 channels, but got 384 channels instead

In [9]:
# Week 3 Day 6 — U-Net Architecture
# Goal: Build complete U-Net with Dice Loss

import torch
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print("✓ Setup complete")

# Cell 2 — Encoder and Decoder blocks (corrected)

class EncoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.relu  = nn.ReLU()
        self.pool  = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        skip = x
        x = self.pool(x)
        return x, skip


class DecoderBlock(nn.Module):
    def __init__(self, x_channels, skip_channels, out_channels):
        super().__init__()
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv1 = nn.Conv2d(x_channels + skip_channels, out_channels,
                               kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels,
                               kernel_size=3, padding=1)
        self.relu  = nn.ReLU()

    def forward(self, x, skip):
        x = self.upsample(x)
        x = torch.cat([x, skip], dim=1)
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        return x

print("✓ EncoderBlock and DecoderBlock classes defined")

# Cell 3 — Complete U-Net (corrected)

class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()

        # Encoder
        self.enc1 = EncoderBlock(in_channels, 16)
        self.enc2 = EncoderBlock(16, 32)
        self.enc3 = EncoderBlock(32, 64)
        self.enc4 = EncoderBlock(64, 128)

        # Bottleneck
        self.bottleneck_conv1 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bottleneck_conv2 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.relu = nn.ReLU()

        # Decoder — x_channels, skip_channels, out_channels
        self.dec4 = DecoderBlock(x_channels=256, skip_channels=128, out_channels=128)
        self.dec3 = DecoderBlock(x_channels=128, skip_channels=64,  out_channels=64)
        self.dec2 = DecoderBlock(x_channels=64,  skip_channels=32,  out_channels=32)
        self.dec1 = DecoderBlock(x_channels=32,  skip_channels=16,  out_channels=16)

        # Final output layer
        self.final_conv = nn.Conv2d(16, out_channels, kernel_size=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # Encoder
        x, skip1 = self.enc1(x)
        print(f"After enc1: x={x.shape}, skip1={skip1.shape}")
        x, skip2 = self.enc2(x)
        print(f"After enc2: x={x.shape}, skip2={skip2.shape}")
        x, skip3 = self.enc3(x)
        print(f"After enc3: x={x.shape}, skip3={skip3.shape}")
        x, skip4 = self.enc4(x)
        print(f"After enc4: x={x.shape}, skip4={skip4.shape}")

        # Bottleneck
        x = self.relu(self.bottleneck_conv1(x))
        x = self.relu(self.bottleneck_conv2(x))
        print(f"After bottleneck: x={x.shape}")

        # Decoder
        x = self.dec4(x, skip4)
        print(f"After dec4: {x.shape}")
        x = self.dec3(x, skip3)
        print(f"After dec3: {x.shape}")
        x = self.dec2(x, skip2)
        print(f"After dec2: {x.shape}")
        x = self.dec1(x, skip1)
        print(f"After dec1: {x.shape}")

        # Final output
        x = self.final_conv(x)
        x = self.sigmoid(x)
        print(f"Final output: {x.shape}")

        return x

print("✓ UNet class defined")

# Cell 4 — Test the U-Net with a fake CT slice

model = UNet(in_channels=1, out_channels=1).to(device)

ct_input = torch.zeros(1, 1, 128, 128).to(device)
print(f"Input shape: {ct_input.shape}\n")

with torch.no_grad():
    output = model(ct_input)

print(f"\n✓ Final segmentation output shape: {output.shape}")

# Cell 5 — Dice Loss
# Measures overlap between predicted and true segmentation

class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth

    def forward(self, prediction, target):
        # Flatten both tensors to 1D
        prediction = prediction.view(-1)
        target = target.view(-1)

        # Calculate overlap
        intersection = (prediction * target).sum()

        # Dice formula
        dice = (2. * intersection + self.smooth) / (
            prediction.sum() + target.sum() + self.smooth
        )

        # Loss = 1 - Dice (so lower loss = better overlap)
        return 1 - dice


print("── Testing Dice Loss ──")

dice_loss = DiceLoss()

print("\nScenario 1 — Perfect prediction:")
prediction = torch.tensor([1.0, 1.0, 0.0, 0.0])
target     = torch.tensor([1.0, 1.0, 0.0, 0.0])
loss = dice_loss(prediction, target)
print(f"Prediction: {prediction}")
print(f"Target:     {target}")
print(f"Dice Loss:  {loss.item():.4f}")

print("\nScenario 2 — No overlap at all:")
prediction = torch.tensor([1.0, 1.0, 0.0, 0.0])
target     = torch.tensor([0.0, 0.0, 1.0, 1.0])
loss = dice_loss(prediction, target)
print(f"Prediction: {prediction}")
print(f"Target:     {target}")
print(f"Dice Loss:  {loss.item():.4f}")

print("\nScenario 3 — Partial overlap:")
prediction = torch.tensor([1.0, 1.0, 1.0, 0.0])
target     = torch.tensor([1.0, 1.0, 0.0, 0.0])
loss = dice_loss(prediction, target)
print(f"Prediction: {prediction}")
print(f"Target:     {target}")
print(f"Dice Loss:  {loss.item():.4f}")

print("\n── Test on real U-Net output shape ──")
fake_prediction = torch.rand(1, 1, 128, 128).to(device)  # random 0-1 values
fake_mask       = torch.randint(0, 2, (1, 1, 128, 128)).float().to(device)  # random 0s and 1s

loss = dice_loss(fake_prediction, fake_mask)
print(f"Prediction shape: {fake_prediction.shape}")
print(f"Mask shape:       {fake_mask.shape}")
print(f"Dice Loss:        {loss.item():.4f}")

Device: cuda
✓ Setup complete
✓ EncoderBlock and DecoderBlock classes defined
✓ UNet class defined
Input shape: torch.Size([1, 1, 128, 128])

After enc1: x=torch.Size([1, 16, 64, 64]), skip1=torch.Size([1, 16, 128, 128])
After enc2: x=torch.Size([1, 32, 32, 32]), skip2=torch.Size([1, 32, 64, 64])
After enc3: x=torch.Size([1, 64, 16, 16]), skip3=torch.Size([1, 64, 32, 32])
After enc4: x=torch.Size([1, 128, 8, 8]), skip4=torch.Size([1, 128, 16, 16])
After bottleneck: x=torch.Size([1, 256, 8, 8])
After dec4: torch.Size([1, 128, 16, 16])
After dec3: torch.Size([1, 64, 32, 32])
After dec2: torch.Size([1, 32, 64, 64])
After dec1: torch.Size([1, 16, 128, 128])
Final output: torch.Size([1, 1, 128, 128])

✓ Final segmentation output shape: torch.Size([1, 1, 128, 128])
── Testing Dice Loss ──

Scenario 1 — Perfect prediction:
Prediction: tensor([1., 1., 0., 0.])
Target:     tensor([1., 1., 0., 0.])
Dice Loss:  0.0000

Scenario 2 — No overlap at all:
Prediction: tensor([1., 1., 0., 0.])
Target:  